<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# EduTrack Analytics
## Notebook 2 — Data Cleaning & Feature Engineering
### 📝 VERSION APPRENANT

> **Instructions :** Complète les cellules marquées `# TODO`.  
> Les blocs `MÉTHODE` t'expliquent l'approche attendue.  
> Les blocs `🧠 Tes observations` sont à remplir après exécution.

| | |
|---|---|
| **Prérequis** | Notebook 1 complété |
| **Niveau** | Intermédiaire |
| **Outils** | Python — pandas, numpy, matplotlib |
| **Durée estimée** | 3h à 4h |

---
> Le Notebook 1 a identifié **6 anomalies** dans les 5 fichiers. Ce notebook les corrige toutes systématiquement, puis construit les variables analytiques clés du projet. La qualité du feature engineering conditionne directement la performance du modèle ML du NB5.

---
## Étape 1 — Imports et chargement

> **MÉTHODE — Pourquoi recharger les CSV depuis zéro ?**
>
> On recharge les 5 CSV depuis zéro pour partir d'un état propre et reproductible. Ne jamais continuer depuis l'état d'un notebook précédent sans recharger — les transformations intermédiaires pourraient avoir modifié les données en mémoire.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os, sys

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

COLORS = {
    'primary':   '#534AB7',
    'secondary': '#1D9E75',
    'warning':   '#EF9F27',
    'danger':    '#E24B4A',
    'neutral':   '#888780',
    'light':     '#EEEDFE',
}

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F9F9F8',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.size':        11,
})

# ── Détection Colab / Local ──────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/elearning_analytics/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Environnement : {"Colab" if IN_COLAB else "Local"}')
print(f'📁 Dossier       : {SAVE_PATH}')
print('Configuration chargée ✅')

In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/data/'

df_parc  = pd.read_csv(BASE_URL + 'parcours.csv')
df_app   = pd.read_csv(BASE_URL + 'apprenants.csv',   parse_dates=['date_inscription'])
df_inscr = pd.read_csv(BASE_URL + 'inscriptions.csv', parse_dates=['date_inscription', 'date_fin_reelle'])
df_sess  = pd.read_csv(BASE_URL + 'sessions.csv',     parse_dates=['date_session'])
df_paie  = pd.read_csv(BASE_URL + 'paiements.csv',    parse_dates=['date_paiement'])

DATE_REF = pd.Timestamp('2024-06-30')

print('Chargement OK')
for name, df in [('parcours', df_parc), ('apprenants', df_app),
                  ('inscriptions', df_inscr), ('sessions', df_sess),
                  ('paiements', df_paie)]:
    print(f'  {name:<20} {len(df):>7,} lignes')

---
## Étape 2 — Nettoyage : apprenants.csv

### MÉTHODE
On suit un ordre logique :
1. D'abord les doublons (réduire le volume avant les autres opérations)
2. Ensuite les valeurs aberrantes (corriger les incohérences métier)
3. Enfin les valeurs manquantes (imputer ou signaler)

`drop_duplicates(keep='first')` conserve la première occurrence — dans un contexte CRM, la première inscription est généralement la référence.

In [ ]:
# 1. Supprimer les doublons email (keep='first')
# TODO : df_app = df_app.drop_duplicates(...)

# 2. Corriger les âges aberrants par imputation à la médiane
# TODO : calculer mediane_age sur les âges valides [16-80]
# TODO : créer mask_age et remplacer

# 3. Remplir les pays null
# TODO : df_app['pays'] = df_app['pays'].fillna(...)

print(f'Apprenants après nettoyage : {len(df_app):,}')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 3 — Nettoyage : inscriptions.csv

### MÉTHODE
Les progressions > 100% sont une incohérence métier : un apprenant ne peut pas avoir complété plus que 100% d'un parcours. On plafonne à 100 — c'est la correction la plus conservative (minimise la perte d'information).

In [ ]:
# 1. Plafonner les progressions > 100% à 100
# TODO : df_inscr.loc[..., 'progression_pct'] = 100.0

# 2. Supprimer les inscriptions orphelines
# (apprenants supprimés en étape précédente)
# TODO : filtrer df_inscr sur apprenant_id.isin(df_app)

print('Distribution statuts :')
print(df_inscr['statut'].value_counts())

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 4 — Nettoyage : sessions.csv et paiements.csv

### MÉTHODE
Les durées négatives dans sessions sont des erreurs techniques (problème de timestamp au logout avant le login). On les supprime car une durée de session invalide fausserait le calcul de l'`engagement_score`.

In [ ]:
# 1. Supprimer les sessions avec durée <= 0
# TODO : df_sess = df_sess[df_sess['duree_minutes'] > 0]

# 2. Supprimer les sessions orphelines
# TODO : filtrer df_sess sur apprenant_id.isin(df_app)

# 3. Supprimer les paiements négatifs
# TODO : df_paie = df_paie[df_paie['montant_fcfa'] > 0]

print(f'Sessions restantes  : {len(df_sess):,}')
print(f'Paiements restants  : {len(df_paie):,}')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 5 — Feature Engineering

### 5.1 — Variables temporelles

### MÉTHODE
Les variables temporelles sont essentielles pour le ML sur des données d'apprentissage. `jours_depuis_inscription` capture la maturation de l'inscription : un apprenant inscrit depuis 30 jours à 20% de progression est dans une situation très différente de quelqu'un inscrit depuis 300 jours à 20%. `mois_inscription` sera encodé en sin/cos dans le NB5 pour capturer la saisonnalité cyclique.

In [ ]:
# TODO : créer jours_depuis_inscription = (DATE_REF - date_inscription).dt.days
# TODO : créer mois_inscription = date_inscription.dt.month
# TODO : créer est_inscription_weekend (1 si sam/dim, 0 sinon)

print('Variables temporelles créées :')
print(f'  jours_depuis_inscription : mean={df_inscr["jours_depuis_inscription"].mean():.0f}')
print(f'  inscriptions le weekend  : {df_inscr["est_inscription_weekend"].mean()*100:.1f}%')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

### 5.2 — Engagement Score

### MÉTHODE
L'`engagement_score` est une feature composite qui mesure l'implication globale de l'apprenant dans son parcours. La formule pondère :
- **40%** sur le nombre de sessions (fréquence)
- **40%** sur la durée totale en heures (profondeur)
- **20%** sur le nombre de semaines actives (régularité)

La normalisation 0-100 par MinMax permet de comparer des apprenants de parcours de durées différentes (6 semaines vs 20 semaines).

In [ ]:
# Agrégation sessions par (apprenant, parcours)
sess_agg = df_sess.groupby(['apprenant_id', 'parcours_id']).agg(
    nb_sessions      = ('session_id',   'count'),
    duree_totale_min = ('duree_minutes', 'sum'),
    nb_jours_actifs  = ('date_session',  lambda x: x.dt.date.nunique())
).reset_index()

# Semaines actives
sess_sem = (
    df_sess.groupby(['apprenant_id', 'parcours_id'])['date_session']
    .apply(lambda x: x.dt.isocalendar().week.nunique())
    .reset_index()
    .rename(columns={'date_session': 'nb_semaines_actives'})
)

sess_agg = sess_agg.merge(sess_sem, on=['apprenant_id', 'parcours_id'], how='left')
df_inscr = df_inscr.merge(sess_agg, on=['apprenant_id', 'parcours_id'], how='left')
df_inscr[['nb_sessions','duree_totale_min','nb_jours_actifs','nb_semaines_actives']] = \
    df_inscr[['nb_sessions','duree_totale_min','nb_jours_actifs','nb_semaines_actives']].fillna(0)

# TODO : calculer engagement_score normalisé 0-100
# Formule : 40% nb_sessions + 40% duree_totale_heures + 20% nb_semaines_actives
# Normaliser chaque composante par son max avant d'appliquer les poids
max_sess  = df_inscr['nb_sessions'].max()
max_duree = (df_inscr['duree_totale_min'] / 60).max()
max_sem   = df_inscr['nb_semaines_actives'].max()

df_inscr['engagement_score'] = # TODO

print(f'engagement_score - mean : {df_inscr["engagement_score"].mean():.2f}')
print(f'engagement_score - max  : {df_inscr["engagement_score"].max():.2f}')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

### 5.3 — Jours d'inactivité

### MÉTHODE
`nb_jours_inactif` mesure depuis quand l'apprenant ne s'est pas connecté. C'est la feature la plus prédictive du décrochage — un apprenant absent depuis 30 jours sur un parcours de 8 semaines a très probablement abandonné. Pour les apprenants sans aucune session, on utilise `jours_depuis_inscription` car ils n'ont jamais commencé — c'est le cas le plus extrême d'inactivité.

In [ ]:
# Dernière session par (apprenant, parcours)
last_sess = (
    df_sess.groupby(['apprenant_id', 'parcours_id'])['date_session']
    .max().reset_index()
    .rename(columns={'date_session': 'last_session'})
)
df_inscr = df_inscr.merge(last_sess, on=['apprenant_id', 'parcours_id'], how='left')

# TODO : calculer nb_jours_inactif = (DATE_REF - last_session).dt.days
# TODO : pour les apprenants sans session, utiliser jours_depuis_inscription
# Indice : .fillna(df_inscr['jours_depuis_inscription'])

print(f'nb_jours_inactif - mean : {df_inscr["nb_jours_inactif"].mean():.1f} jours')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

### 5.4 — Variable cible : at_risk_dropout

### MÉTHODE
La définition de la variable cible est le choix le plus important du projet. On adopte une définition inclusive qui capture deux types de risque :
1. **Risque avéré** : l'apprenant a déjà abandonné (statut='Abandonne')
2. **Risque précoce** : l'apprenant est inactif depuis > 21 jours, a une faible progression (< 80%) et n'a pas encore terminé

Le seuil de 21 jours correspond à 3 semaines — soit 37% d'un parcours de 8 semaines. Un apprenant qui n'ouvre pas l'application pendant 3 semaines consécutives est statistiquement peu susceptible de terminer.

In [ ]:
# TODO : définir at_risk_dropout selon 2 conditions :
# cond1 : statut == 'Abandonne'
# cond2 : inactif > 21j ET progression < 80% ET pas Terminé
cond1 = # TODO
cond2 = # TODO

df_inscr['at_risk_dropout'] = (cond1 | cond2).astype(int)

print('Distribution at_risk_dropout :')
print(df_inscr['at_risk_dropout'].value_counts())
print(f"Taux at_risk : {df_inscr['at_risk_dropout'].mean()*100:.1f}%")

# Visualisation de la distribution
fig, ax = plt.subplots(figsize=(7, 4))
counts = df_inscr['at_risk_dropout'].value_counts()
bars = ax.bar(['Pas à risque (0)', 'À risque (1)'], counts.values,
              color=[COLORS['secondary'], COLORS['danger']])
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{val:,}\n({val/len(df_inscr)*100:.1f}%)', ha='center', fontweight='bold')
ax.set_title('Distribution de la variable cible at_risk_dropout', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}at_risk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 6 — Export de la table analytique

### MÉTHODE
On sélectionne uniquement les colonnes nécessaires pour le ML et le dashboard. Les colonnes brutes redondantes (`date_fin_reelle`, `last_session`) sont exclues pour ne pas créer de **leakage** dans le NB5.

> **Leakage :** une feature qui contient implicitement la réponse à la question qu'on cherche à prédire. Par exemple, inclure `date_fin_reelle` dans les features du modèle serait du leakage car cette date n'existe que pour les apprenants qui ont terminé ou abandonné — le modèle apprendrait à mémoriser, pas à généraliser.

In [ ]:
COLS_ANALYTICS = [
    'inscription_id', 'apprenant_id', 'parcours_id', 'date_inscription',
    'statut', 'progression_pct', 'csat', 'jours_depuis_inscription',
    'mois_inscription', 'est_inscription_weekend', 'nb_sessions',
    'duree_totale_min', 'nb_semaines_actives', 'engagement_score',
    'nb_jours_inactif', 'at_risk_dropout'
]
df_analytics = df_inscr[COLS_ANALYTICS].copy()
df_analytics.to_csv(f'{SAVE_PATH}inscriptions_analytics.csv', index=False)
print(f'✅ inscriptions_analytics.csv exporté : {df_analytics.shape}')
print(f'   Nulls restants : {df_analytics.isnull().sum().sum()}')

---
## Bilan du Notebook 2

| Action | Lignes | Impact |
|---|---|---|
| Doublons email supprimés | 3 | df_app : 4 500 → 4 497 |
| Âges aberrants corrigés | 5 | Imputation par médiane (27 ans) |
| Pays null remplis | 2 | 'Non renseigné' |
| Progressions plafonnées | 4 | 100% maximum |
| Sessions supprimées | 67 | 57 075 → 57 008 |
| Paiements négatifs supprimés | 3 | 8 703 → 8 700 |
| **Taux at_risk_dropout** | **47.9%** | **3 092 / 6 456** |

**Fichier exporté :** `inscriptions_analytics.csv` (6 456 lignes × 16 colonnes) — disponible dans `SAVE_PATH`

| Feature | Type | Rôle ML |
|---|---|---|
| `engagement_score` | Float 0-100 | Feature composite (fréquence + durée + régularité) |
| `nb_jours_inactif` | Int | Feature la plus prédictive du décrochage |
| `at_risk_dropout` | Binaire 0/1 | **Variable cible** du modèle ML |

**Pour le NB3 :** Utiliser `inscriptions_analytics.csv` avec DuckDB pour les analyses SQL et les KPIs de performance.

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.